In [ ]:
import cv2
import numpy as np
import csv
import datetime
import time
from collections import Counter
from tensorflow.keras.models import load_model

# Load the trained grayscale model
model = load_model("facial_emotion_model.h5")

# Emotion labels - must match training order
emotion_labels = ['angry', 'neutral', 'sleep', 'stress']

# Initialize webcam
cap = cv2.VideoCapture(0)

# Load Haar cascade for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Emotion tracking per 10s window
emotion_buffer = []
confidence_buffer = []
start_time = time.time()

# Use full file path to ensure write location
output_path = "emotion_log_10s.csv"

# Open CSV using context manager so it's flushed and saved properly
with open(output_path, mode='w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['Timestamp', 'Dominant Emotion', 'Confidence'])

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

        for (x, y, w, h) in faces:
            face_gray = gray[y:y+h, x:x+w]
            face_gray = cv2.resize(face_gray, (48, 48))

            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            face_gray = clahe.apply(face_gray)

            face_gray = face_gray.astype('float32') / 255.0
            face_gray = np.reshape(face_gray, (1, 48, 48, 1))


            prediction = model.predict(face_gray)
            confidence = np.max(prediction)

            if confidence > 0.4:
                emotion = emotion_labels[np.argmax(prediction)]
                emotion_buffer.append(emotion)
                confidence_buffer.append(confidence)

                # Draw on screen
                color = (0, 255, 0) if confidence > 0.7 else (0, 255, 255)
                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
                cv2.putText(frame, f"{emotion} ({confidence:.2f})", (x, y-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # Every 10s, log most common emotion
        if time.time() - start_time >= 10:
            if emotion_buffer:
                most_common = Counter(emotion_buffer).most_common(1)[0][0]
                avg_conf = np.mean([c for e, c in zip(emotion_buffer, confidence_buffer) if e == most_common])
                timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                # Write to CSV and flush
                csv_writer.writerow([timestamp, most_common, f"{avg_conf:.2f}"])
                csv_file.flush()
                print(f"Saved: {timestamp}, {most_common}, {avg_conf:.2f}")
            else:
                print("No emotion detected in the last 10 seconds.")

            emotion_buffer.clear()
            confidence_buffer.clear()
            start_time = time.time()

        cv2.imshow("Emotion Detection", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


No emotion detected in the last 10 seconds.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Saved: 2025-07-20 12:57:40, sleep, 0.56
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Saved: 2025-07-20 12:57:50, sleep, 0.65
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Saved: 2025-07-20 12:58:00, sleep, 0.43
No emotion detected in the last 10 se